## tl;dr
Dos fichas laborales parciales, no costo patronal calibrado. MX tiene pago anunciado por km y vales mensuales; US conserva referencias truckload históricas separadas de OEWS/ECEC. Cero adopciones. La carga social administrativa no carga por sí sola al conductor.

## Context & Methods
Piloto de terceros D2D caja seca MX/US, con intercambio. Reutiliza el contrato existente; no cambia fórmulas activas.
### Key Assumptions
Las cantidades, períodos y unidades públicas mantienen su alcance. Los ejemplos de exposición/YTD son sintéticos, no inputs propuestos para bases. No se suma salario de encabezado conflictivo ni se transfieren factores entre cohortes.
Fuentes y localizadores están en el JSON compañero. Python estándar, sin red, kernel Jupyter o paquetes externos. Ejecutar desde la raíz del worktree; tablas pequeñas, sin gráfico porque son comprobaciones aritméticas/no series comparables.

In [1]:
import json, math, hashlib
from pathlib import Path
research = Path("docs/research")
d = json.loads((research / "B2_LABOR_COMPENSATION_BRIDGE_2026-09-29.json").read_text(encoding="utf-8"))
assert d["adoptedValues"] == 0 and not d["calibrated"] and not d["quoteReady"]
for original in d["unchangedFiles"]:
    assert hashlib.sha256((research / original["file"]).read_bytes()).hexdigest() == original["sha256"]
assert len(d["administration"]["roles"]) == 7
assert all(v is None for v in d["outputs"].values())
print("Context, four immutable artifacts, seven roles and unknown outputs: PASS")

Context, four immutable artifacts, seven roles and unknown outputs: PASS


## Results
### Remuneración y denominadores
Sólo transformación de tarifa anunciada. El costo de empleador completo y FX siguen desconocidos. El caso 8,000/10,000 km es sintético para mostrar la diferencia entre km pagados y físicos.

In [2]:
rate = d["mxDriver"]["offer"]["ratePerPaidLoadedKm"]
assert math.isclose(rate * d["kmPerMile"], d["mxDriver"]["normalizedWageOnlyMxnPerPaidMile"], abs_tol=1e-10)
assert d["mxDriver"]["offer"]["headerSalary"]["inferredMonthlySalary"] is None
assert d["mxDriver"]["driverUsdPerKm"] is None
partial_synthetic = rate * 8000 + d["mxDriver"]["offer"]["monthlyGroceryVouchers"]
assert math.isclose(partial_synthetic / 10000, 2.023, abs_tol=1e-12)
assert not math.isclose(partial_synthetic / 10000, rate + 3430 / 8000, abs_tol=1e-9)
print(f"Anuncio MX: {rate * d['kmPerMile']:.6f} MXN/milla pagada, sólo sueldo")
print(f"Caso sintético: subtotal {partial_synthetic:.2f} / 10000 km vehículo = {partial_synthetic / 10000:.3f}; no costo patronal")
print("5 checks MX: PASS")

Anuncio MX: 3.379614 MXN/milla pagada, sólo sueldo
Caso sintético: subtotal 20230.00 / 10000 km vehículo = 2.023; no costo patronal
5 checks MX: PASS


### Beneficios US: participación versus recargo
ECEC junio 2026, transporte/almacenaje todas las ocupaciones. No se combina con OEWS 2025 ni ATRI 2024. Diferencia de un centavo entre componentes redondeados y total publicado conservada.

In [3]:
r = d["usSectorBenefitContrast"]
assert math.isclose(r["wages"] + r["benefits"], 49.85, abs_tol=1e-10)
assert math.isclose(r["total"] - r["wages"] - r["benefits"], 0.01, abs_tol=1e-10)
assert math.isclose(sum(r["benefitComponents"].values()), r["benefits"], abs_tol=1e-10)
assert math.isclose(r["benefitToWageRatio"], r["benefits"] / r["wages"], abs_tol=1e-12)
assert math.isclose(r["benefitToTotalRatio"], r["benefits"] / r["total"], abs_tol=1e-12)
assert r["benefitToWageRatio"] > r["benefitToTotalRatio"]
assert not r["transferredToDriverOrAdmin"] and r["legallyRequiredAlreadyInsideBenefits"]
h = d["usDriver"]["historicalTruckloadRateRange"]
assert math.isclose(h["minUsdPerKm"], 0.790 / d["kmPerMile"], abs_tol=1e-12)
assert math.isclose(h["maxUsdPerKm"], 1.058 / d["kmPerMile"], abs_tol=1e-12)
print(f"ECEC salario {r['wages']:.2f}, beneficios {r['benefits']:.2f}, total publicado {r['total']:.2f} USD/hora trabajada")
print(f"Beneficios/salario {100*r['benefitToWageRatio']:.2f}%; beneficios/total {100*r['benefitToTotalRatio']:.2f}%")
print(f"ATRI histórico separado: {h['minUsdPerKm']:.6f} a {h['maxUsdPerKm']:.6f} USD/km; no intervalo de confianza/costo completo")
print("9 checks US: PASS")

ECEC salario 32.95, beneficios 16.90, total publicado 49.86 USD/hora trabajada
Beneficios/salario 51.29%; beneficios/total 33.89%
ATRI histórico separado: 0.490884 a 0.657412 USD/km; no intervalo de confianza/costo completo
9 checks US: PASS


### Cargas federales: límite anual incremental
IRS 2026 aporta reglas, no el factor efectivo del carrier. Ejemplos YTD/corriente sintéticos. No se suma la retención del trabajador ni el Additional Medicare al patrón. FUTA efectivo, SUI y workers compensation siguen desconocidos.

In [4]:
rules = d["usFederalRules"]
def ss_period(current_covered_wage, previous_ytd_covered_wage):
    return rules["employerSocialSecurityRate"] * min(current_covered_wage, max(rules["socialSecurityAnnualTaxableWageCap"] - previous_ytd_covered_wage, 0))
assert math.isclose(ss_period(10000, 180000), 279, abs_tol=1e-10)
assert ss_period(10000, 184500) == 0
assert math.isclose(ss_period(10000, 0), 620, abs_tol=1e-10)
assert math.isclose(10000 * rules["employerMedicareRate"], 145, abs_tol=1e-10)
assert rules["additionalMedicareEmployerShare"] == 0
assert rules["effectiveFutaRate"] is None and not rules["maximumCreditAssumed"]
assert rules["stateUnemploymentRate"] is None and rules["workersCompensationRate"] is None
print("Synthetic current covered wage / previous YTD / employer SS")
for current, prior in [(10000,180000),(10000,184500),(10000,0)]:
    print(current, prior, f"{ss_period(current,prior):.2f}")
print("7 checks tax/allocation boundaries: PASS")

Synthetic current covered wage / previous YTD / employer SS
10000 180000 279.00
10000 184500 0.00
10000 0 620.00
7 checks tax/allocation boundaries: PASS


## Takeaways
21 comprobaciones analíticas más checks de contexto/hashes. No adopción ni declaraciones fiscales. El siguiente corte ensambla las fichas económicas parciales por familia y subtotales elegibles, sin rellenar faltantes con cero ni abrir otra ronda genérica de salarios. El notebook requiere ejecutar y revisar su presentación guardada; no confundir esas comprobaciones con aceptación operativa.